# Automated Warehouse Robot Controller

This notebook outlines the design and implementation of a multi-agent control system for coordinating a fleet of warehouse robots. The system coordinates multiple robots to retrieve shelves and deliver them to packing stations without collisions or deadlocks. Various **Multi-Agent Pathfinding (MAPF)** techniques, including **Independent A***, **Cooperative A***, **Hill Climbing Optimization**, and **Conflict-Based Search (CBS)**, are employed to find collision-free paths that minimize both makespan and flowtime. The project includes comprehensive performance evaluation, deadlock detection, and visual simulation with heatmap analysis of warehouse congestion patterns.

## Data Collection & Research
We will be working on the Standard dataset by the time we collect a local one 

There are multiple choices for the standard dataset:
- **Small Grid**: 63*161
- **Medium Grid**: 84*170
- **Large Grid**: 123*321
- **Very Larg Grid**: 165*340

### Grid Envirenment Class
This class represents the warehouse as a grid-based environment that manages boundaries, obstacles, and valid robot movements.

In [ ]:
class GridEnvironment:
    """
    Represents a 2D grid-based warehouse environment.
    
    Attributes:
        grid: 2D array where True = free space, False = obstacle (shelf/wall)
        height: Number of rows
        width: Number of columns
        
    Methods provide obstacle checking, neighbor finding, and visualization.
    """

    def __init__(self, filename):
        """ Load grid from file """
        self.grid = self.load_from_file(filename)
        """ Initialize grid with dimensions """
        self.height = len(self.grid)
        self.width = len(self.grid[0]) if self.height > 0 else 0

    def is_valid_position(self, x, y):
        """ Check if position is within grid bounds """
        return 0 <= x < self.width and 0 <= y < self.height

    def is_walkable(self, x, y):
        """ Check if position is walkable (not obstacle)"""
        return self.is_valid_position(x, y) and self.grid[y][x]

    def get_neighbors(self, x, y):
        """Get adjacent cells (4-directional)"""
        directions = [(0,1), (1,0), (0,-1), (-1,0)]
        neighbors = []

        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if self.is_walkable(nx, ny):
                neighbors.append((nx, ny))

        return neighbors

    def load_from_file(self, filename):
        """ Load grid from file """
        grid = []
        with open(filename, 'r') as f:
            for line in f:
                row = []
                for char in line.strip():
                    if char == '.':
                        row.append(True)
                    elif char == 'T':
                        row.append(False)
                grid.append(row)
        return grid

    def save_to_file(self, filename):
        """ Save grid to file """
        with open(filename, 'w') as f:
            for row in self.grid:
                line = ''.join(['.' if cell else 'T' for cell in row])
                f.write(line + '\n')

    def visualize(self):
        import matplotlib.pyplot as plt
        from matplotlib.colors import ListedColormap
        import numpy as np
        """Visualize the warehouse grid"""
        grid_array = np.array(self.grid, dtype=int)
        
        fig, ax = plt.subplots(figsize=(12, 12))
        
        cmap = ListedColormap(['black', 'white'])
        ax.imshow(grid_array, cmap=cmap, origin='upper', interpolation='nearest')
        
        ax.set_xticks(np.arange(-0.5, self.width, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.height, 1), minor=True)
        
        ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.3, alpha=0.5)
        
        ax.set_xticks(np.arange(0, self.width, 20))
        ax.set_yticks(np.arange(0, self.height, 20))
        

        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(f"Warehouse Grid ({self.width}x{self.height})")
        
        plt.tight_layout()
        plt.show()

print("Grid class defnined")

## Problem Definition

Given N robots at start positions and N goal positions (shelves to retrieve), find a set of
conflict-free paths such that all robots reach their goals in minimal time.

## Problem Formulation
### 1. State Representation

Each state is represented by a dictionary with the following structure:

```python
state = {
    
    # The Robots
    'robots': [
        {
            'id': robot_id,                             # Unique identifier (0, 1, 2, ...)
            'start_position': (x0, y0),                 # Start postion (fixed)
            'goal_position': (gx, gy),                  # Goal position (fixed)
            'at_goal': bool,                            # True if positions[robot_id] == goal_position
            'color': str                                # The color of the robot, used for visualization stuff
        },
        # ... more robots
    ],
    
    # Global state
    'positions': {robot_id: (x, y)},          # Current position of the robot in the grid given its id
    'collisions': count,                      # Cumulative collision count
    'deadlock': bool,                         # True if circular wait detected
    'grid': GridEnvironment                   # Reference to warehouse layout
}
```

### 2. Goal Test

**Primary Goal**:
- No collisions occurred
- No deadlock detected
- All robots at their goal positions


**Success Criteria: (Detail)**
1. All robots reach goals 
2. No collisions/conflicts 
3. No deadlocks 
4. Minimum makespan  (secondary objective)
5. Minimum flowtime  (secondary objective)

### 3. Actions
- All robots move simultaneously, some could wait

- SubActions per robot: 
    - MoveUp: move up if walkable (path = [..., (x, y), (x - 1, y), ...]) 


    - MoveDown: move down if walkable (path = [..., (x, y), (x + 1, y), ...]) 


    - MoveLeft: move left if walkable (path = [..., (x, y), (x, y - 1), ...]) 


    - MoveRight: move right if walkable (path = [..., (x, y), (x, y + 1), ...]) 


    - Wait: stay in current position (path = [..., (x, y), (x, y), ...]) 



### 4. Transition Model
- The new state after applying the action:
    - The new position for each robot is added to its path, if it is a wait just add the last position to the path i.e. [..., (x, y), (x, y), ...]

    - Each robot may reach its goal_position, hence the attribute 'at_goal' may be updated 

    - The positions of the robots is changed

    - The number of collisions in the state is updated

    - The deadlock state is update (a deadlock may occur)

### 5. Path Cost

- Each move costs: 1 time unit
- Each wait costs: 1 time unit


### Initial State


**Properties of Initial State:**
- All robots at distinct start positions
- All start positions are walkable
- No paths planned yet
- No conflicts exist initially
- Time counter starts at 0
- Cost g(n) = 0 (no moves yet)


**Initial State Example:**
```python
initial_state = {
    'robots': [
        {'id': 1, 'start_position': (1, 1), 'goal_position': (8, 8), 'at_goal': False},
        {'id': 2, 'start_position': (8, 1), 'goal_position': (1, 8), 'at_goal': False},
        {'id': 3, 'start_position': (1, 8), 'goal_position': (8, 1), 'at_goal': False},
    ],
    'positions': {1: (1, 1), 2: (8, 1), 3: (1, 8)}
    'collisions': 0,
    'deadlock': False,
    'grid': GridEnvironment(filename)
}
```

- Robot 1: starts at (1,1), goal (8,8)
- Robot 2: starts at (8,1), goal (1,8)
- Robot 3: starts at (1,8), goal (8,1)
- Robots' positions are their starting positions
- No collisions 
- No deadlock
- Grid loaded from the file

### Robot Class

This class represents a single robot in the grid, managing its position, goal, and movement within the environment.

In [ ]:
class Robot :
    """ Represents a single robot """
    def __init__(self, robot_id: int, start_position: tuple, goal_position: tuple, color: str):
        """
        Args:
            grid (GridEnvironment): The grid this robot moves in
            start_pos (tuple): Starting (x, y) position
            goal_pos (tuple): Goal (gx, gy) position
            color (str): Robot color
        """

        self.id = robot_id
        self.start_pos = start_position
        self.goal_pos = goal_position
        self.color = color

    def get_start_position(self) -> tuple:
        """Get start position"""
        return self.start_position

    def get_goal_position(self) -> tuple:
        """Get goal position """
        return self.goal_position

    def get_color(self) -> str:
        """Get color"""
        return self.color
    
print("Robot class defined")

### Node Class
Now, let's define the Node class which will represent states in the search space

In [ ]:
class Node:
    """
    Represents a node in the search tree.
    
    Attributes:
        state: The current state (configuration of all robots)
        parent: Parent node in search tree
        action: Action that led to this node
        g: Cumulative cost (actual path cost from start)
        h: Heuristic value (estimated cost to goal)
        f: Total evaluation cost (g + h)
        depth: Depth in search tree
    """

    def __init__(self, state, parent=None, action=None, g=0, h=0):
        """
        Initialize a search node.
        
        Args:
            state: State configuration (dict of robot positions at each time)
            parent: Parent node
            action: Action to reach this node
            g: Cost from start to this node
            h: Heuristic estimate to goal
        """
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  # Actual cost
        self.h = h  # Heuristic estimate
        self.f = g + h  # f(n) = g(n) + h(n)
        self.depth = 0 if parent is None else parent.depth + 1

    def __hash__(self):
        """
        Make node hashable by hashing the state.
        Since state is a dict, convert to a canonical string form.
        """
        return hash(str(sorted(self.state.items()))) # ⚠️⚠️⚠️⚠️⚠️⚠️ We need to consider the list of robots here, lists are imutable

    def __eq__(self, other):
        """
        Two nodes are equal if their states are identical.
        """
        return self.state == other.state    

    def __gt__(self, other):
        """
        Compare this node with another node based on the evaluation function (f).

        Input Parameters:
            - other: Another Node instance.

        Output:
            - True if this node's f is greater than the other's f, else False.
        """
        return isinstance(other, Node) and self.f > other.f
    
    def __repr__(self):
        return f"Node(depth={self.depth}, g={self.g}, h={self.h}, f={self.f})"
    
print("Class Node definied")

### Candidate Class
Now, we define the Candidate class which will represent a candidate solution in a local search context

In [ ]:
class Candidate:
    """
    Represents a candidate solution in a local search context.

    Attributes:
        state: The specific configuration of the solution (e.g., a tour permutation, queen positions).
        value: The evaluation score of the state (lower is generally better in minimization problems).
    """
    def __init__(self, state, value):
        self.state = state
        self.value = value

    def __repr__(self):
        pass

print("Class Candidate defined")

### Problem Class
Now, let's define the main Problem class that will encapsulate our MAPF problem

In [ ]:
import copy
from itertools import product

class AutomatedWarehouseRobotControllerProblem:
    """
    Encapsulates the Multi-Agent Pathfinding (MAPF) problem for N warehouse robots.

    The problem is defined by an initial state (see state schema in the markdown above).
    This class provides:
      - Goal testing
      - Action generation (all valid joint moves for every robot)
      - State transition (apply a joint action to produce a new state)
      - Node expansion for tree-search algorithms
      - Neighbour generation for local search (Hill Climbing)
      - Makespan / Flowtime metrics
      - Vertex conflict, edge conflict, and deadlock detection
    """

    def __init__(self, initial_state):
        """
        Args:
            initial_state (dict): The starting state of the problem.
                Keys: 'robots', 'positions', 'collisions', 'deadlock', 'grid'
                Each robot dict must have: 'id', 'start_position', 'goal_position',
                'at_goal', 'color'.  A 'path' key is added automatically if absent.
        """
        self.state = copy.deepcopy(initial_state)
        # Initialise path tracking for every robot if not already present
        for robot in self.state['robots']:
            if 'path' not in robot:
                robot['path'] = [self.state['positions'][robot['id']]]

    # ──────────────────────────────────────────────────────────────────────────
    # 1. Goal Test
    # ──────────────────────────────────────────────────────────────────────────

    def is_goal(self, state=None) -> bool:
        """
        Returns True when the state satisfies all primary goal conditions:
          1. All robots are at their goal positions.
          2. No collisions have occurred (collision count == 0).
          3. No deadlock is active.

        Args:
            state (dict | None): State to test. Uses self.state if None.

        Returns:
            bool
        """
        s = state if state is not None else self.state
        if s['collisions'] > 0 or s['deadlock']:
            return False
        return all(robot['at_goal'] for robot in s['robots'])

    # ──────────────────────────────────────────────────────────────────────────
    # 2. Action Generation
    # ──────────────────────────────────────────────────────────────────────────

    def get_valid_actions(self, state) -> list:
        """
        Returns all valid joint actions for the given state.

        A joint action is a dict mapping robot_id → next_(x, y).
        Each robot independently may: Move Up/Down/Left/Right (if walkable) or Wait.
        The Cartesian product of per-robot moves gives all joint actions.

        Edge conflicts (swap moves) are filtered out here so that downstream
        planners never receive an action that causes a head-on collision.

        Args:
            state (dict): Current state.

        Returns:
            list[dict[int, tuple]]: Each element is one joint action.
                Example: [{0: (3,4), 1: (7,2)}, {0: (3,4), 1: (7,3)}, ...]
        """
        grid      = state['grid']
        positions = state['positions']
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]  # R D L U Wait

        per_robot_moves = {}
        for robot in state['robots']:
            rid  = robot['id']
            x, y = positions[rid]
            moves = []
            for dx, dy in directions:
                nx, ny = x + dx, y + dy
                if grid.is_walkable(nx, ny):
                    moves.append((nx, ny))
            per_robot_moves[rid] = moves

        robot_ids  = list(per_robot_moves.keys())
        move_lists = [per_robot_moves[rid] for rid in robot_ids]

        valid_joint_actions = []
        for combo in product(*move_lists):
            joint = dict(zip(robot_ids, combo))
            # Filter out edge (swap) conflicts in the joint action
            if not self._has_edge_conflict_in_action(positions, joint):
                valid_joint_actions.append(joint)

        return valid_joint_actions

    def _has_edge_conflict_in_action(self, positions, joint_action) -> bool:
        """
        Returns True if any two robots swap positions in this joint action
        (i.e. robot A moves from u to v while robot B moves from v to u).

        Args:
            positions    (dict): Current {robot_id: (x,y)}
            joint_action (dict): Proposed {robot_id: (nx,ny)}

        Returns:
            bool
        """
        ids = list(joint_action.keys())
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a, b = ids[i], ids[j]
                if (joint_action[a] == positions[b] and
                        joint_action[b] == positions[a]):
                    return True
        return False

    # ──────────────────────────────────────────────────────────────────────────
    # 3. Transition Model
    # ──────────────────────────────────────────────────────────────────────────

    def apply_action(self, state, action) -> dict:
        """
        Applies a joint action to a state and returns the resulting new state.

        Steps:
          1. Move every robot to its next position (or keep it in place for Wait).
          2. Update each robot's path list and 'at_goal' flag.
          3. Update global 'positions'.
          4. Detect vertex collisions and add to 'collisions' count.
          5. Detect deadlocks and set 'deadlock' flag.

        Args:
            state  (dict): Current state (not mutated).
            action (dict): Joint action {robot_id: (nx, ny)}.

        Returns:
            dict: New state after the action.
        """
        new_state = copy.deepcopy(state)

        # Apply moves
        new_positions = {}
        for robot in new_state['robots']:
            rid        = robot['id']
            next_pos   = action[rid]
            new_positions[rid] = next_pos
            robot['path'].append(next_pos)
            robot['at_goal'] = (next_pos == robot['goal_position'])

        new_state['positions'] = new_positions

        # Vertex conflict detection
        vertex_collisions = self._count_vertex_conflicts(new_positions)
        new_state['collisions'] += vertex_collisions

        # Deadlock detection
        new_state['deadlock'] = self._detect_deadlock(state['positions'], new_positions)

        return new_state

    # ──────────────────────────────────────────────────────────────────────────
    # 4. Node Expansion
    # ──────────────────────────────────────────────────────────────────────────

    def expand_node(self, node) -> list:
        """
        Generates all child nodes of the given search-tree node.

        For each valid joint action from the node's state, a new Node is
        created with:
          - state  = result of apply_action
          - parent = node
          - action = the joint action applied
          - g      = node.g + 1  (each time-step costs 1)
          - h      = sum of Manhattan distances from each robot's new position
                     to its goal (admissible heuristic for the joint problem)

        Args:
            node (Node): The node to expand.

        Returns:
            list[Node]: Child nodes.
        """
        children = []
        for action in self.get_valid_actions(node.state):
            new_state = self.apply_action(node.state, action)
            h = self._joint_heuristic(new_state)
            child = Node(
                state  = new_state,
                parent = node,
                action = action,
                g      = node.g + 1,
                h      = h,
            )
            children.append(child)
        return children

    def _joint_heuristic(self, state) -> int:
        """
        Admissible heuristic for the joint state:
        Sum of Manhattan distances of every robot to its goal.
        (Lower bound on remaining cost, so A* remains optimal.)
        """
        total = 0
        positions = state['positions']
        for robot in state['robots']:
            if not robot['at_goal']:
                rid      = robot['id']
                x,  y    = positions[rid]
                gx, gy   = robot['goal_position']
                total   += abs(x - gx) + abs(y - gy)
        return total

    # ──────────────────────────────────────────────────────────────────────────
    # 5. Neighbour Generation (for Hill Climbing)
    # ──────────────────────────────────────────────────────────────────────────

    def generate_neighbors(self, state) -> list:
        """
        Returns neighbour states for local search (Hill Climbing).

        Each neighbour is obtained by applying exactly ONE valid joint action
        to the given state.  This is the same as the set returned by
        get_valid_actions / apply_action, but wrapped as Candidate objects
        so Hill Climbing can score and compare them directly.

        Args:
            state (dict): Current state.

        Returns:
            list[Candidate]: Each Candidate holds the resulting state and its
                             evaluation value (makespan + flowtime, lower = better).
        """
        neighbours = []
        for action in self.get_valid_actions(state):
            next_state = self.apply_action(state, action)
            value      = self.get_makespan(next_state) + self.get_flowtime(next_state)
            neighbours.append(Candidate(state=next_state, value=value))
        return neighbours

    # ──────────────────────────────────────────────────────────────────────────
    # 6. Metrics
    # ──────────────────────────────────────────────────────────────────────────

    def get_makespan(self, state=None) -> int:
        """
        Makespan = the number of steps taken by the robot that finishes last.

        Formally: max over all robots of len(robot['path']) - 1.
        A robot that starts at its goal has makespan 0.

        Args:
            state (dict | None): State to measure. Uses self.state if None.

        Returns:
            int: Makespan (0 if no paths exist).
        """
        s = state if state is not None else self.state
        lengths = [len(r.get('path', [])) - 1 for r in s['robots']]
        return max(lengths) if lengths else 0

    def get_flowtime(self, state=None) -> int:
        """
        Flowtime = sum of individual travel times across all robots.

        Formally: sum over all robots of len(robot['path']) - 1.

        Args:
            state (dict | None): State to measure. Uses self.state if None.

        Returns:
            int: Flowtime (0 if no paths exist).
        """
        s = state if state is not None else self.state
        return sum(len(r.get('path', [])) - 1 for r in s['robots'])

    # ──────────────────────────────────────────────────────────────────────────
    # 7. Conflict & Deadlock Detection
    # ──────────────────────────────────────────────────────────────────────────

    def _count_vertex_conflicts(self, positions: dict) -> int:
        """
        Counts the number of vertex conflicts in a single time step.

        A vertex conflict occurs when two or more robots occupy the same cell
        at the same time.  Every pair that shares a cell counts as 1 conflict.

        Args:
            positions (dict): {robot_id: (x, y)} after the joint move.

        Returns:
            int: Number of conflicting pairs.
        """
        pos_list = list(positions.values())
        conflicts = 0
        for i in range(len(pos_list)):
            for j in range(i + 1, len(pos_list)):
                if pos_list[i] == pos_list[j]:
                    conflicts += 1
        return conflicts

    def has_vertex_conflict(self, positions: dict) -> bool:
        """
        Returns True if any two robots occupy the same cell.

        Args:
            positions (dict): {robot_id: (x, y)}

        Returns:
            bool
        """
        return self._count_vertex_conflicts(positions) > 0

    def has_edge_conflict(self, prev_positions: dict, next_positions: dict) -> bool:
        """
        Returns True if any two robots swap positions between consecutive steps.

        Robot A moves u → v while Robot B moves v → u constitutes an edge
        (swap) conflict regardless of whether they physically pass through the
        same cell.

        Args:
            prev_positions (dict): {robot_id: (x,y)} at time t.
            next_positions (dict): {robot_id: (x,y)} at time t+1.

        Returns:
            bool
        """
        ids = list(next_positions.keys())
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a, b = ids[i], ids[j]
                if (next_positions[a] == prev_positions[b] and
                        next_positions[b] == prev_positions[a]):
                    return True
        return False

    def _detect_deadlock(self, prev_positions: dict, next_positions: dict) -> bool:
        """
        Detects a *circular wait* deadlock among robots.

        A deadlock is declared when a group of robots form a cycle where each
        robot in the cycle is waiting for the cell currently occupied by the
        next robot in the cycle — i.e. every robot's next position equals
        another robot's current position AND no robot actually moves.

        Detection algorithm (cycle in the "wait-for" graph):
          - Build a directed graph: robot A → robot B if A's next position
            equals B's current position and A did not successfully move.
          - Run DFS to find any cycle in this graph.

        Args:
            prev_positions (dict): {robot_id: (x,y)} at time t.
            next_positions (dict): {robot_id: (x,y)} at time t+1.

        Returns:
            bool: True if a circular deadlock is detected.
        """
        # Invert: position → robot_id at time t
        pos_to_id = {pos: rid for rid, pos in prev_positions.items()}

        # Build wait-for graph: A waits-for B if A did not move and
        # A's intended next position is where B currently sits.
        wait_for = {}
        for rid, next_pos in next_positions.items():
            if next_pos == prev_positions[rid]:          # robot did not move
                if next_pos in pos_to_id:
                    blocker = pos_to_id[next_pos]
                    if blocker != rid:
                        wait_for[rid] = blocker

        if not wait_for:
            return False

        # DFS cycle detection
        visited = set()
        in_stack = set()

        def has_cycle(node):
            visited.add(node)
            in_stack.add(node)
            neighbour = wait_for.get(node)
            if neighbour is not None:
                if neighbour not in visited:
                    if has_cycle(neighbour):
                        return True
                elif neighbour in in_stack:
                    return True
            in_stack.discard(node)
            return False

        for node in list(wait_for.keys()):
            if node not in visited:
                if has_cycle(node):
                    return True
        return False

print("Class AutomatedWarehouseRobotControllerProblem defined")

### Search Algorithms

In [ ]:
import heapq
import copy
from itertools import product as iproduct

# ══════════════════════════════════════════════════════════════════════════════
#  A_Star  —  Base class
# ══════════════════════════════════════════════════════════════════════════════

class A_Star:
    """
    Base A* class.  Provides:
      • Manhattan-distance heuristic (admissible for 4-directional grids).
      • Space-Time A* for a single robot (_astar_single_robot).
      • Conflict helpers: vertex-conflict check, edge-conflict check,
        and reservation-table builder.
    Subclasses override solve() to implement their specific strategy.
    """

    def __init__(self, problem, strategy: str = "base"):
        """
        Args:
            problem  : AutomatedWarehouseRobotControllerProblem instance.
            strategy : label string, overridden in subclasses.
        """
        self.problem  = problem
        self.strategy = strategy

    # ── Heuristic ─────────────────────────────────────────────────────────────

    def heuristic(self, x: int, y: int, gx: int, gy: int) -> int:
        """
        Manhattan distance from (x,y) to goal (gx,gy).
        Admissible because every step costs exactly 1 unit.
        """
        return abs(x - gx) + abs(y - gy)

    # ── Conflict helpers ──────────────────────────────────────────────────────

    def _check_vertex_conflict(self, pos: tuple, t: int, reservation_table: set) -> bool:
        """
        Returns True if cell `pos` at time `t` is already reserved
        by a previously planned robot.

        A vertex conflict: two robots at the same (x,y) at the same time t.
        """
        return (pos[0], pos[1], t) in reservation_table

    def _check_edge_conflict(
        self,
        pos_from: tuple,
        pos_to: tuple,
        t: int,
        reservation_table: set,
    ) -> bool:
        """
        Returns True if the move pos_from → pos_to at time step t would
        create a swap (edge) conflict with any reserved path.

        Swap conflict: robot A moves u→v while robot B moves v→u.
        Detected by checking:
          - pos_to was reserved at t-1  (the other robot was there before)
          - pos_from is reserved at t   (the other robot moves into our origin)
        """
        other_was_at_to   = (pos_to[0],   pos_to[1],   t - 1) in reservation_table
        other_goes_to_src = (pos_from[0], pos_from[1], t    ) in reservation_table
        return other_was_at_to and other_goes_to_src

    def _build_reservation_table(self, paths: list) -> set:
        """
        Converts a list of planned paths into a reservation table (set of
        (x, y, t) tuples).

        After a robot reaches its goal it is assumed to stay there, so the
        goal cell is reserved for all future time steps up to max path length.

        Args:
            paths : list of lists, each inner list is [(x0,y0), (x1,y1), …]

        Returns:
            set of (x, y, t) tuples.
        """
        if not paths:
            return set()
        max_len = max(len(p) for p in paths)
        table   = set()
        for path in paths:
            for t, pos in enumerate(path):
                table.add((pos[0], pos[1], t))
            if path:
                goal = path[-1]
                for t in range(len(path), max_len + 1):
                    table.add((goal[0], goal[1], t))
        return table

    # ── Core single-robot Space-Time A* ──────────────────────────────────────

    def _astar_single_robot(
        self,
        robot,
        grid,
        reservation_table: set,
        start_time: int = 0,
        max_time:   int = 500,
    ):
        """
        Space-Time A* for a SINGLE robot.

        Search state : (x, y, t) — position AND discrete time.
        Heuristic    : Manhattan distance to goal (time-independent).
        Conflict checks are performed against `reservation_table` at every
        expansion step, so the resulting path is guaranteed free of both
        vertex and edge conflicts with all previously planned robots.

        Args:
            robot             : object with .start_pos (x,y) and .goal_pos (x,y).
            grid              : GridEnvironment.
            reservation_table : set of (x,y,t) already claimed.
            start_time        : time offset for the first node.
            max_time          : upper bound on search depth.

        Returns:
            list of (x,y) from start to goal (inclusive), or None if not found.
        """
        sx, sy = robot.start_pos
        gx, gy = robot.goal_pos

        # Open list: (f, g, x, y, t)
        h0 = self.heuristic(sx, sy, gx, gy)
        open_heap = [(h0, 0, sx, sy, start_time)]

        came_from = {(sx, sy, start_time): None}
        g_score   = {(sx, sy, start_time): 0}

        directions = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]   # 4 moves + wait

        while open_heap:
            f, g, x, y, t = heapq.heappop(open_heap)

            # Goal check
            if (x, y) == (gx, gy):
                return self._reconstruct_path(came_from, (x, y, t))

            if t >= max_time:
                continue

            # Skip stale open-list entries
            if g > g_score.get((x, y, t), float('inf')):
                continue

            nt = t + 1
            for dx, dy in directions:
                nx, ny = x + dx, y + dy

                # Grid walkability
                if not grid.is_walkable(nx, ny):
                    continue

                # Vertex conflict
                if self._check_vertex_conflict((nx, ny), nt, reservation_table):
                    continue

                # Edge (swap) conflict — only for actual moves, not wait
                if (dx, dy) != (0, 0):
                    if self._check_edge_conflict((x, y), (nx, ny), nt, reservation_table):
                        continue

                new_g     = g + 1
                new_state = (nx, ny, nt)
                if new_g < g_score.get(new_state, float('inf')):
                    g_score[new_state]   = new_g
                    came_from[new_state] = (x, y, t)
                    h = self.heuristic(nx, ny, gx, gy)
                    heapq.heappush(open_heap, (new_g + h, new_g, nx, ny, nt))

        return None   # no path found within max_time

    def _reconstruct_path(self, came_from: dict, goal_state: tuple) -> list:
        """Walk came_from back to the start and return path as [(x,y), …]."""
        path    = []
        current = goal_state
        while current is not None:
            x, y, _ = current
            path.append((x, y))
            current = came_from[current]
        path.reverse()
        return path

    def solve(self):
        """Override in subclasses."""
        raise NotImplementedError("Subclasses must implement solve().")


# ══════════════════════════════════════════════════════════════════════════════
#  IndependentAStar  —  Baseline (no collision avoidance during planning)
# ══════════════════════════════════════════════════════════════════════════════

class IndependentAStar(A_Star):
    """
    Independent A* — each robot plans its shortest path on the static grid,
    completely ignoring every other robot.

    Because robots are unaware of each other, vertex and edge conflicts are
    common in crowded maps.  All conflicts are *detected* post-planning and
    reported, but NOT resolved — this is the baseline expected to deadlock
    or collide frequently.

    result = IndependentAStar(problem).solve()

    Result keys
    ───────────
    'paths'      : {robot_id: [(x,y),…] | None}
    'collisions' : list of conflict dicts
    'success'    : True iff all paths found AND zero collisions
    'makespan'   : max path length
    'flowtime'   : sum of path lengths
    """

    def __init__(self, problem):
        super().__init__(problem, strategy="independent")

    def solve(self) -> dict:
        """
        Plan each robot independently (empty reservation table).
        Detect collisions between all planned paths afterwards.
        """
        state  = self.problem.state
        grid   = state['grid']

        paths = {}
        for robot_data in state['robots']:
            rid          = robot_data['id']
            robot_proxy  = _RobotProxy(robot_data['start_position'],
                                       robot_data['goal_position'])
            path         = self._astar_single_robot(
                               robot=robot_proxy,
                               grid=grid,
                               reservation_table=set(),
                           )
            paths[rid] = path

        collisions  = self._detect_all_collisions(paths)
        valid_paths = [p for p in paths.values() if p is not None]
        makespan    = max(len(p) for p in valid_paths) if valid_paths else 0
        flowtime    = sum(len(p) for p in valid_paths)

        return {
            'paths'      : paths,
            'collisions' : collisions,
            'success'    : all(p is not None for p in paths.values())
                           and len(collisions) == 0,
            'makespan'   : makespan,
            'flowtime'   : flowtime,
        }

    def _detect_all_collisions(self, paths: dict) -> list:
        """
        Post-planning collision detector.

        Pads shorter paths (robot stays at goal) then checks every pair at
        every shared time step for vertex and edge conflicts.

        Returns a list of dicts, each describing one conflict.
        """
        collisions = []
        robot_ids  = [rid for rid, p in paths.items() if p is not None]
        if len(robot_ids) < 2:
            return collisions

        max_len = max(len(paths[rid]) for rid in robot_ids)

        def padded(rid):
            p = paths[rid]
            return p + [p[-1]] * (max_len - len(p))

        pp = {rid: padded(rid) for rid in robot_ids}

        for i in range(len(robot_ids)):
            for j in range(i + 1, len(robot_ids)):
                a, b = robot_ids[i], robot_ids[j]
                pa, pb = pp[a], pp[b]
                for t in range(max_len):
                    if pa[t] == pb[t]:
                        collisions.append({'type': 'vertex', 'time': t,
                                           'robot_a': a, 'robot_b': b,
                                           'pos': pa[t]})
                    elif t > 0 and pa[t] == pb[t-1] and pa[t-1] == pb[t]:
                        collisions.append({'type': 'edge', 'time': t,
                                           'robot_a': a, 'robot_b': b,
                                           'pos_a': pa[t-1], 'pos_b': pb[t-1]})
        return collisions


# ══════════════════════════════════════════════════════════════════════════════
#  CooperativeAStar  —  Sequential planning with reservation table
# ══════════════════════════════════════════════════════════════════════════════

class CooperativeAStar(A_Star):
    """
    Cooperative A* (Space-Time A* / Hierarchical).

    Robots are planned one at a time in priority order.  Each planned path
    is added to a shared *reservation table* before the next robot plans,
    so later robots treat earlier paths as dynamic (time-indexed) obstacles.

    Guarantees:
      • No vertex conflicts between any two successfully planned robots.
      • No edge (swap) conflicts between any two successfully planned robots.
      • Completeness is NOT guaranteed in very dense maps (a blocked robot
        returns None for its path).

    result = CooperativeAStar(problem).solve()
    result = CooperativeAStar(problem, priority_order=[2,0,1]).solve()

    Result keys
    ───────────
    'paths'      : {robot_id: [(x,y),…] | None}
    'collisions' : []  (empty by construction)
    'success'    : True iff every robot has a valid path
    'makespan'   : max path length
    'flowtime'   : sum of path lengths
    'order'      : planning order used
    """

    def __init__(self, problem, priority_order: list = None):
        """
        Args:
            problem        : AutomatedWarehouseRobotControllerProblem.
            priority_order : optional list of robot ids.  Defaults to the
                             order in problem.state['robots'].
        """
        super().__init__(problem, strategy="cooperative")
        self.priority_order = priority_order

    def solve(self) -> dict:
        """
        Plan robots sequentially, growing the reservation table after each.
        """
        state      = self.problem.state
        grid       = state['grid']
        robot_by_id = {r['id']: r for r in state['robots']}

        order = (self.priority_order
                 if self.priority_order is not None
                 else [r['id'] for r in state['robots']])

        reservation_table = set()
        planned_paths     = []
        paths             = {}

        for rid in order:
            rd          = robot_by_id[rid]
            robot_proxy = _RobotProxy(rd['start_position'], rd['goal_position'])
            path = self._astar_single_robot(
                       robot=robot_proxy,
                       grid=grid,
                       reservation_table=reservation_table,
                   )
            paths[rid] = path
            if path is not None:
                planned_paths.append(path)
                reservation_table = self._build_reservation_table(planned_paths)

        valid_paths = [p for p in paths.values() if p is not None]
        makespan    = max(len(p) for p in valid_paths) if valid_paths else 0
        flowtime    = sum(len(p) for p in valid_paths)

        return {
            'paths'      : paths,
            'collisions' : [],
            'success'    : all(p is not None for p in paths.values()),
            'makespan'   : makespan,
            'flowtime'   : flowtime,
            'order'      : order,
        }


# ══════════════════════════════════════════════════════════════════════════════
#  _RobotProxy  (internal helper — not part of the public API)
# ══════════════════════════════════════════════════════════════════════════════

class _RobotProxy:
    """
    Minimal stand-in for Robot, providing .start_pos and .goal_pos.
    Used so A* methods can work from raw state dicts without constructing
    full Robot objects (which have a getter bug in the original class).
    """
    __slots__ = ('start_pos', 'goal_pos')
    def __init__(self, start_pos, goal_pos):
        self.start_pos = start_pos
        self.goal_pos  = goal_pos


# ══════════════════════════════════════════════════════════════════════════════
#  Hill_Climbing  —  Local search optimisation over Cooperative A* solution
# ══════════════════════════════════════════════════════════════════════════════

class Hill_Climbing:
    """
    Hill Climbing optimiser for MAPF solutions.

    Takes a *valid* solution (dict of paths, e.g. from CooperativeAStar) and
    tries to reduce its makespan by iteratively attempting to shorten the
    longest path through single-step trimming and wait-removal.

    Algorithm (steepest-ascent variant):
      while improvement found:
        for each robot with path length > optimal (Manhattan):
          try removing each consecutive pair of identical positions (waits)
          try trimming the last wait before the goal
          if any edit reduces makespan → accept it, restart outer loop

    Conflict safety:
      After every candidate edit, the modified path set is checked for
      vertex and edge conflicts.  Only conflict-free edits are accepted.

    Usage:
        coop_result = CooperativeAStar(problem).solve()
        hc          = Hill_Climbing(problem)
        improved    = hc.optimize(coop_result)
    """

    def __init__(self, problem):
        self.problem = problem

    # ── Public entry point ────────────────────────────────────────────────────

    def optimize(self, solution: dict, max_iterations: int = 200) -> dict:
        """
        Attempt to reduce the makespan of `solution` by removing redundant
        wait steps from individual robot paths.

        Args:
            solution       : result dict from IndependentAStar or CooperativeAStar.
            max_iterations : safety cap on the number of improvement attempts.

        Returns:
            dict with same keys as the input solution, with (possibly shorter)
            paths and updated makespan / flowtime.
        """
        paths = {rid: list(p) for rid, p in solution['paths'].items()
                 if p is not None}

        improved = True
        iteration = 0
        while improved and iteration < max_iterations:
            improved  = False
            iteration += 1

            # Find the robot with the longest path
            longest_rid = max(paths, key=lambda r: len(paths[r]))

            for rid in sorted(paths, key=lambda r: -len(paths[r])):
                path = paths[rid]
                # Try removing each wait step (consecutive duplicate positions)
                for i in range(len(path) - 1, 0, -1):
                    if path[i] == path[i - 1]:           # this is a wait step
                        candidate = path[:i] + path[i+1:]
                        test_paths = {r: paths[r] for r in paths}
                        test_paths[rid] = candidate
                        if self._is_conflict_free(test_paths):
                            paths[rid] = candidate
                            improved   = True
                            break        # restart from longest-path check
                if improved:
                    break

        valid  = [p for p in paths.values()]
        result = dict(solution)
        result['paths']    = paths
        result['makespan'] = max(len(p) for p in valid) if valid else 0
        result['flowtime'] = sum(len(p) for p in valid)
        return result

    # ── Conflict checker ──────────────────────────────────────────────────────

    def _is_conflict_free(self, paths: dict) -> bool:
        """
        Returns True if no vertex or edge conflict exists across all paths.
        Paths are padded to equal length (robot stays at goal).
        """
        ids     = list(paths.keys())
        max_len = max(len(paths[r]) for r in ids) if ids else 0

        def padded(rid):
            p = paths[rid]
            return p + [p[-1]] * (max_len - len(p))

        pp = {rid: padded(rid) for rid in ids}

        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a, b = ids[i], ids[j]
                pa, pb = pp[a], pp[b]
                for t in range(max_len):
                    if pa[t] == pb[t]:
                        return False
                    if t > 0 and pa[t] == pb[t-1] and pa[t-1] == pb[t]:
                        return False
        return True


# ══════════════════════════════════════════════════════════════════════════════
#  Conflict_Based  —  CBS (Bonus: two-level search)
# ══════════════════════════════════════════════════════════════════════════════

class Conflict_Based:
    """
    Conflict-Based Search (CBS) — Bonus Algorithm.

    Two-level search for optimal MAPF solutions:

    High level — Conflict Tree (CT):
      • Each CT node holds a set of constraints and a path for each robot.
      • The root node has no constraints; each robot's path is its optimal
        unconstrained path (from individual A*).
      • A CT node is expanded by finding the first conflict, then creating
        two children: one that forbids robot A from cell C at time t, and
        one that forbids robot B from cell C at time t.
      • The CT is searched with a best-first strategy (min cost = sum of
        individual path lengths = flowtime).

    Low level — Constrained A*:
      • Re-plans a single robot's path respecting the robot's personal
        constraint set (forbidden (x,y,t) tuples) AND the shared reservation
        table of already-resolved robots (for speed, we pass constraints
        directly into the Space-Time A* reservation table).

    CBS is optimal (finds the minimum-flowtime solution) and complete.
    It is much slower than Cooperative A* on large instances but handles
    cases where Cooperative A* fails.

    Usage:
        result = Conflict_Based(problem).solve()

    Result keys
    ───────────
    'paths'      : {robot_id: [(x,y),…] | None}
    'collisions' : []  (optimal conflict-free solution)
    'success'    : True iff all robots have a path
    'makespan'   : max path length
    'flowtime'   : sum of path lengths (minimised by CBS)
    'nodes_expanded': number of CT nodes popped from the open list
    """

    def __init__(self, problem):
        self.problem  = problem
        # Re-use Space-Time A* from A_Star base class
        self._astar   = A_Star(problem, strategy="cbs-low-level")

    # ── Public solve ──────────────────────────────────────────────────────────

    def solve(self, max_ct_nodes: int = 2000) -> dict:
        """
        Run CBS.

        Args:
            max_ct_nodes : safety cap on CT nodes expanded (prevents OOM on
                           very large instances).

        Returns:
            dict with keys listed in the class docstring.
        """
        state      = self.problem.state
        grid       = state['grid']
        robot_ids  = [r['id'] for r in state['robots']]
        robot_map  = {r['id']: r for r in state['robots']}

        # ── Root node: plan each robot with NO constraints ────────────────────
        root_constraints = {rid: set() for rid in robot_ids}
        root_paths = {}
        for rid in robot_ids:
            rd    = robot_map[rid]
            proxy = _RobotProxy(rd['start_position'], rd['goal_position'])
            path  = self._astar._astar_single_robot(
                        robot=proxy, grid=grid,
                        reservation_table=set())
            root_paths[rid] = path

        root_cost = sum(len(p) for p in root_paths.values() if p is not None)

        # CT open list: (cost, unique_id, constraints, paths)
        # unique_id breaks ties in the heap (Python can't compare dicts)
        _uid      = [0]
        def next_uid():
            _uid[0] += 1
            return _uid[0]

        open_ct = [(root_cost, next_uid(), root_constraints, root_paths)]
        nodes_expanded = 0

        while open_ct and nodes_expanded < max_ct_nodes:
            cost, _, constraints, paths = heapq.heappop(open_ct)
            nodes_expanded += 1

            # ── Find the first conflict in this CT node ───────────────────────
            conflict = self._find_first_conflict(paths)
            if conflict is None:
                # No conflicts → this is the optimal solution
                valid    = [p for p in paths.values() if p is not None]
                makespan = max(len(p) for p in valid) if valid else 0
                flowtime = sum(len(p) for p in valid)
                return {
                    'paths'          : paths,
                    'collisions'     : [],
                    'success'        : all(p is not None for p in paths.values()),
                    'makespan'       : makespan,
                    'flowtime'       : flowtime,
                    'nodes_expanded' : nodes_expanded,
                }

            # ── Expand: one child per robot involved in the conflict ──────────
            ctype, t, rid_a, rid_b, pos = conflict
            for rid_to_constrain in (rid_a, rid_b):
                new_constraints = copy.deepcopy(constraints)
                # Add forbidden (x,y,t) for this robot
                new_constraints[rid_to_constrain].add((pos[0], pos[1], t))

                # Re-plan only the constrained robot
                rd    = robot_map[rid_to_constrain]
                proxy = _RobotProxy(rd['start_position'], rd['goal_position'])
                reservation = new_constraints[rid_to_constrain]
                new_path = self._astar._astar_single_robot(
                               robot=proxy, grid=grid,
                               reservation_table=reservation)

                if new_path is None:
                    continue    # this branch has no solution, prune it

                new_paths = dict(paths)
                new_paths[rid_to_constrain] = new_path
                new_cost  = sum(len(p) for p in new_paths.values()
                                if p is not None)
                heapq.heappush(open_ct, (new_cost, next_uid(),
                                         new_constraints, new_paths))

        # Exhausted CT without finding a solution
        return {
            'paths'          : {rid: None for rid in robot_ids},
            'collisions'     : [{'type': 'no_solution'}],
            'success'        : False,
            'makespan'       : 0,
            'flowtime'       : 0,
            'nodes_expanded' : nodes_expanded,
        }

    # ── Conflict finder ───────────────────────────────────────────────────────

    def _find_first_conflict(self, paths: dict):
        """
        Scan paths time-step by time-step and return the first conflict found.

        Returns a tuple (type, t, rid_a, rid_b, pos) or None if conflict-free.
          type : 'vertex' or 'edge'
          t    : time step of the conflict
          pos  : (x,y) of the conflict cell (for edge conflicts: the cell
                 robot_a is trying to enter)
        """
        ids     = [rid for rid, p in paths.items() if p is not None]
        max_len = max(len(paths[r]) for r in ids) if ids else 0

        def padded(rid):
            p = paths[rid]
            return p + [p[-1]] * (max_len - len(p))

        pp = {rid: padded(rid) for rid in ids}

        for t in range(max_len):
            for i in range(len(ids)):
                for j in range(i + 1, len(ids)):
                    a, b = ids[i], ids[j]
                    pa, pb = pp[a], pp[b]
                    if pa[t] == pb[t]:
                        return ('vertex', t, a, b, pa[t])
                    if t > 0 and pa[t] == pb[t-1] and pa[t-1] == pb[t]:
                        return ('edge', t, a, b, pa[t])
        return None

print("A_Star, IndependentAStar, CooperativeAStar defined")
print("Hill_Climbing defined")
print("Conflict_Based (CBS) defined")

### Comparative Evaluation

We compare **Independent A*** vs **Cooperative A*** across fleet sizes (5 and 20 robots) using a procedurally generated warehouse grid, measuring:
- **Success Rate** — how often each algorithm produces a collision-free solution
- **Deadlock Rate** — how often Independent A* deadlocks
- **Makespan** — time until the last robot reaches its goal
- **Flowtime** — total steps summed across all robots

In [ ]:
import random
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.animation as animation
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# ── Synthetic grid generator ──────────────────────────────────────────────────

class SyntheticGridEnvironment:
    """
    Generates a simple warehouse-like grid in memory (no file needed).
    Layout: open aisles separated by shelf rows (obstacles).
    """
    def __init__(self, width=30, height=20, obstacle_density=0.25, seed=42):
        random.seed(seed)
        self.width  = width
        self.height = height
        # Start fully open
        self.grid = [[True] * width for _ in range(height)]
        # Add random internal obstacles (simulate shelves)
        for y in range(1, height - 1):
            for x in range(1, width - 1):
                if random.random() < obstacle_density:
                    self.grid[y][x] = False
        # Ensure border is walls
        for x in range(width):
            self.grid[0][x] = False
            self.grid[height-1][x] = False
        for y in range(height):
            self.grid[y][0] = False
            self.grid[y][width-1] = False

    def is_valid_position(self, x, y):
        return 0 <= x < self.width and 0 <= y < self.height

    def is_walkable(self, x, y):
        return self.is_valid_position(x, y) and self.grid[y][x]

    def get_neighbors(self, x, y):
        directions = [(0,1),(1,0),(0,-1),(-1,0)]
        return [(x+dx, y+dy) for dx,dy in directions
                if self.is_walkable(x+dx, y+dy)]

    def walkable_cells(self):
        return [(x, y)
                for y in range(self.height)
                for x in range(self.width)
                if self.grid[y][x]]

    def visualize(self, title="Warehouse Grid"):
        grid_array = np.array(self.grid, dtype=int)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(grid_array, cmap='gray_r', origin='upper')
        ax.set_title(title)
        ax.set_xlabel("X"); ax.set_ylabel("Y")
        plt.tight_layout(); plt.show()


def make_initial_state(grid, n_robots, seed=0):
    """
    Randomly assign n_robots non-overlapping start and goal positions
    on walkable cells.
    """
    random.seed(seed)
    walkable = grid.walkable_cells()
    chosen   = random.sample(walkable, n_robots * 2)
    starts   = chosen[:n_robots]
    goals    = chosen[n_robots:]

    colors = ['red','blue','green','orange','purple','cyan','magenta',
              'yellow','lime','pink','teal','brown','navy','olive',
              'coral','violet','gold','silver','indigo','turquoise']

    robots = []
    for i, (start, goal) in enumerate(zip(starts, goals)):
        robots.append({
            'id'            : i,
            'start_position': start,
            'goal_position' : goal,
            'at_goal'       : False,
            'color'         : colors[i % len(colors)],
            'path'          : [start],
        })

    return {
        'robots'    : robots,
        'positions' : {i: starts[i] for i in range(n_robots)},
        'collisions': 0,
        'deadlock'  : False,
        'grid'      : grid,
    }


# ── Run a single experiment ───────────────────────────────────────────────────

def run_experiment(n_robots, grid, seed=42):
    state   = make_initial_state(grid, n_robots, seed=seed)
    problem = AutomatedWarehouseRobotControllerProblem(state)

    # Independent A*
    t0      = time.perf_counter()
    ind_res = IndependentAStar(problem).solve()
    ind_t   = time.perf_counter() - t0

    # Cooperative A*
    t0       = time.perf_counter()
    coop_res = CooperativeAStar(problem).solve()
    coop_t   = time.perf_counter() - t0

    return {
        'n_robots'       : n_robots,
        'ind_success'    : ind_res['success'],
        'ind_collisions' : len(ind_res['collisions']),
        'ind_makespan'   : ind_res['makespan'],
        'ind_flowtime'   : ind_res['flowtime'],
        'ind_time'       : round(ind_t, 4),
        'coop_success'   : coop_res['success'],
        'coop_collisions': len(coop_res['collisions']),
        'coop_makespan'  : coop_res['makespan'],
        'coop_flowtime'  : coop_res['flowtime'],
        'coop_time'      : round(coop_t, 4),
    }


# ── Run evaluation ────────────────────────────────────────────────────────────

grid   = SyntheticGridEnvironment(width=30, height=20, obstacle_density=0.20, seed=7)
fleet_sizes = [5, 10, 20]
results = [run_experiment(n, grid, seed=n*3) for n in fleet_sizes]

# ── Print results table ───────────────────────────────────────────────────────

print(f"{'Robots':>7} | {'Ind Success':>11} | {'Ind Collis':>10} | "
      f"{'Ind Make':>8} | {'Ind Flow':>8} | {'Coop Success':>12} | "
      f"{'Coop Make':>9} | {'Coop Flow':>9}")
print("-" * 95)
for r in results:
    print(f"{r['n_robots']:>7} | {str(r['ind_success']):>11} | {r['ind_collisions']:>10} | "
          f"{r['ind_makespan']:>8} | {r['ind_flowtime']:>8} | {str(r['coop_success']):>12} | "
          f"{r['coop_makespan']:>9} | {r['coop_flowtime']:>9}")

### Deliverables

The cells below produce all required visual deliverables:
1. **Grid visualisation** — warehouse layout
2. **Animated simulation** — 10 robots moving simultaneously (Cooperative A*)
3. **Congestion heatmap** — which corridors are most used
4. **Comparison bar charts** — Independent A* vs Cooperative A* metrics

In [ ]:
# ── Deliverable 1: Warehouse Grid Visualisation ──────────────────────────────

grid.visualize(title="Synthetic Warehouse Grid (30×20)")

# ── Deliverable 2: Animated Simulation (10 robots, Cooperative A*) ────────────

N_ANIM   = 10
state_10 = make_initial_state(grid, N_ANIM, seed=99)
prob_10  = AutomatedWarehouseRobotControllerProblem(state_10)
coop_10  = CooperativeAStar(prob_10).solve()

paths_10 = coop_10['paths']          # {robot_id: [(x,y), ...]}
robots_10 = state_10['robots']
colors_10 = {r['id']: r['color'] for r in robots_10}

# Pad paths to equal length
max_t = max(len(p) for p in paths_10.values() if p)

def pad(path, length):
    return path + [path[-1]] * (length - len(path))

padded_10 = {rid: pad(p, max_t) for rid, p in paths_10.items() if p}

fig_anim, ax_anim = plt.subplots(figsize=(10, 7))
grid_array = np.array(grid.grid, dtype=int)
ax_anim.imshow(grid_array, cmap='gray_r', origin='upper', alpha=0.7)

# Draw goal markers
for r in robots_10:
    gx, gy = r['goal_position']
    ax_anim.plot(gx, gy, marker='*', markersize=10,
                 color=colors_10[r['id']], alpha=0.5)

robot_dots = {}
for rid in padded_10:
    sx, sy = padded_10[rid][0]
    dot, = ax_anim.plot(sx, sy, 'o', markersize=8,
                        color=colors_10[rid], zorder=5)
    robot_dots[rid] = dot

time_text = ax_anim.text(0.02, 0.95, '', transform=ax_anim.transAxes,
                          fontsize=10, color='white',
                          bbox=dict(facecolor='black', alpha=0.5))

def update(frame):
    for rid, dot in robot_dots.items():
        x, y = padded_10[rid][frame]
        dot.set_data([x], [y])
    time_text.set_text(f't = {frame}')
    return list(robot_dots.values()) + [time_text]

ani = animation.FuncAnimation(fig_anim, update, frames=max_t,
                               interval=200, blit=True)
ax_anim.set_title(f"Cooperative A* — {N_ANIM} robots (stars = goals)")
ax_anim.set_xlabel("X"); ax_anim.set_ylabel("Y")
plt.tight_layout()
plt.show()
print(f"Animation: {max_t} time steps, success={coop_10['success']}, "
      f"makespan={coop_10['makespan']}, flowtime={coop_10['flowtime']}")

In [ ]:
# ── Deliverable 3: Congestion Heatmap ────────────────────────────────────────

heatmap = np.zeros((grid.height, grid.width), dtype=float)
for rid, path in paths_10.items():
    if path:
        for (x, y) in path:
            heatmap[y][x] += 1

# Mask obstacles so they appear dark
obstacle_mask = np.array([[0 if grid.grid[y][x] else np.nan
                            for x in range(grid.width)]
                           for y in range(grid.height)])

fig_hm, ax_hm = plt.subplots(figsize=(11, 7))
# Draw obstacle layer
ax_hm.imshow(np.where(np.isnan(obstacle_mask), 0, obstacle_mask),
             cmap='Greys', origin='upper', vmin=0, vmax=1, alpha=0.4)
# Draw heatmap (only walkable cells)
heat_masked = np.where(np.array(grid.grid), heatmap, np.nan)
im = ax_hm.imshow(heat_masked, cmap='hot', origin='upper', alpha=0.85)
plt.colorbar(im, ax=ax_hm, label='Total robot visits (all time steps)')
ax_hm.set_title("Congestion Heatmap — Cooperative A* (10 robots)")
ax_hm.set_xlabel("X"); ax_hm.set_ylabel("Y")
plt.tight_layout(); plt.show()

In [ ]:
# ── Deliverable 4: Comparison Bar Charts ─────────────────────────────────────

labels     = [f"{r['n_robots']} robots" for r in results]
x          = np.arange(len(labels))
width      = 0.35

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Independent A* vs Cooperative A* — Performance Comparison",
             fontsize=13, fontweight='bold')

# (a) Number of collisions
ax = axes[0]
ax.bar(x - width/2, [r['ind_collisions'] for r in results],
       width, label='Independent A*', color='tomato')
ax.bar(x + width/2, [r['coop_collisions'] for r in results],
       width, label='Cooperative A*',  color='steelblue')
ax.set_title("Collision Count")
ax.set_ylabel("Collisions")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.legend()

# (b) Makespan
ax = axes[1]
ax.bar(x - width/2, [r['ind_makespan'] for r in results],
       width, label='Independent A*', color='tomato')
ax.bar(x + width/2, [r['coop_makespan'] for r in results],
       width, label='Cooperative A*',  color='steelblue')
ax.set_title("Makespan (steps until last robot done)")
ax.set_ylabel("Time Steps")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.legend()

# (c) Flowtime
ax = axes[2]
ax.bar(x - width/2, [r['ind_flowtime'] for r in results],
       width, label='Independent A*', color='tomato')
ax.bar(x + width/2, [r['coop_flowtime'] for r in results],
       width, label='Cooperative A*',  color='steelblue')
ax.set_title("Flowtime (total steps summed)")
ax.set_ylabel("Total Steps")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.legend()

plt.tight_layout(); plt.show()

# Summary
print("\n=== Summary ===")
for r in results:
    print(f"\n{r['n_robots']} robots:")
    print(f"  Independent A*  → success={r['ind_success']}, "          f"collisions={r['ind_collisions']}, makespan={r['ind_makespan']}, "          f"flowtime={r['ind_flowtime']}, time={r['ind_time']}s")
    print(f"  Cooperative A*  → success={r['coop_success']}, "          f"collisions={r['coop_collisions']}, makespan={r['coop_makespan']}, "          f"flowtime={r['coop_flowtime']}, time={r['coop_time']}s")